# Thermal expansion workflow with pheasy and a machine-learned potential

This notebook computes the thermal expansion of MgO with the `CTEMaker`
workflow. The forces come from MACE-OMAT-0-medium, so no DFT is needed.

## Background

The workflow computes the thermal expansion tensor from mode Grüneisen tensors.
The thermal stress of each phonon mode is its heat capacity times its Grüneisen
tensor. The strain that relaxes the summed thermal stress is the thermal
expansion:

$$\alpha = \frac{S}{N_q V} \sum_{q\nu} c_{q\nu} \gamma_{q\nu}$$

Here $S$ is the elastic compliance, $c_{q\nu}$ and $\gamma_{q\nu}$ are the heat
capacity and Grüneisen tensor of each mode, $N_q$ is the number of q-points and
$V$ is the cell volume.

The flow has four steps:

1. The structure is converted to the standard primitive cell and relaxed tightly.
2. The pheasy phonon workflow fits the second- and third-order force constants
   with LASSO. It uses randomly displaced supercells with 0.03 Å displacements.
3. The elastic workflow fits the elastic tensor on the same relaxed structure.
4. phono3py computes the mode Grüneisen tensors on a 12x12x12 q-point mesh, and
   the thermal expansion follows from the formula above.

Steps 2 and 3 do not depend on each other, so a workflow manager can run them
at the same time. The same workflow runs with VASP as
`atomate2.vasp.flows.cte.CTEMaker`.

## Installation

The `pheasy` and `ase` extras are needed, plus MACE and the Materials Project
client:

```
pip install 'atomate2[pheasy,ase]'
pip install 'mace-torch>=0.3.16' mp-api
```

The `pheasy` extra pulls in pheasy, phono3py and ALM. ALM is compiled from
source. The hiPhive tutorial and the atomate2 VASP documentation list what the
build needs.

## The potential

MACE-OMAT-0-medium downloads once and caches under `~/.cache/mace`. It is
released under the Academic Software License.

`float64` is worth the cost here. The third-order force constants come from
small force differences between displaced supercells, where `float32` noise is
not negligible.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

from mace.calculators import mace_mp  # noqa: E402

CALC_KWARGS = {"model": "medium-omat-0", "device": "cpu", "default_dtype": "float64"}

_ = mace_mp(**CALC_KWARGS)  # downloads and caches on first use

## One calculator for all jobs

Every force field job builds its own calculator. `run_locally` runs all of
them in this one Python process, and the workflow has a few hundred jobs.
Loading MACE for each of them grows the memory by about 150 MB per job. This
cell makes the jobs share one calculator. It is only needed when many jobs run
in one process, not with a workflow manager.

In [ ]:
from ase.calculators.calculator import Calculator

import atomate2.forcefields.utils as ff_utils

_ase_calculator = ff_utils.ase_calculator
_calculators: dict[str, Calculator] = {}


def shared_ase_calculator(calculator_meta: object, **kwargs: object) -> Calculator:
    """Build each calculator once and reuse it for every job."""
    key = repr((calculator_meta, sorted(kwargs.items())))
    if key not in _calculators:
        _calculators[key] = _ase_calculator(calculator_meta, **kwargs)
    _calculators[key].reset()
    return _calculators[key]


ff_utils.ase_calculator = shared_ase_calculator

## The structure

The primitive cell of MgO (mp-1265) comes from the Materials Project. Set your
key first with `export MP_API_KEY=...`.

The Materials Project serves this cell rotated away from the cubic axes, and
the pheasy fits fail for such a cell. `CTEMaker` therefore converts the input to
the standard primitive cell before the relaxation. This is the default,
`use_symmetrized_structure="primitive"`.

In [ ]:
from mp_api.client import MPRester

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")

with MPRester() as mpr:
    structure = mpr.get_structure_by_material_id("mp-1265")
structure.lattice

## Building the workflow

`CTEMaker.from_force_field_name` sets one force field for the relaxation, the
phonons and the elastic tensor. Any MACE model routes through `mace_mp`, and
the model name in `calculator_kwargs` selects the weights.

Everything else stays at the workflow defaults. The supercells have
`min_length=12` Å, which gives 128 atoms for MgO. The third-order force
constants use the one-shot fit, and the temperatures run from 0 K to 1000 K in
steps of 10 K.

In [ ]:
from atomate2.forcefields.flows.cte import CTEMaker

maker = CTEMaker.from_force_field_name("MACE-MP-0", calculator_kwargs=CALC_KWARGS)
flow = maker.make(structure)
flow.draw_graph().show()

## Running the workflow

The flow has a little over 200 jobs, most of them force calculations on the
displaced supercells. On one Perlmutter CPU node the whole run took about 45
minutes. Most of that time goes into the pheasy fit of the third-order force
constants. Expect longer on a laptop.

`create_folders=True` is needed, because the thermal expansion job reads the
force constants from the folder of the pheasy fit.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

## Results

The output is a `CTEDocument`. It holds the thermal expansion tensor at each
temperature for each fit method. The value at 0 K is zero, since the heat
capacity is zero there. MgO is cubic, so the three diagonal components are
equal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

(result,) = doc.results
temperatures = np.array(doc.temperatures)
alpha = np.array(result.thermal_expansion)
i300 = int(np.argmin(np.abs(temperatures - 300)))

fig, ax = plt.subplots()
ax.plot(temperatures, alpha[:, 0, 0] * 1e6)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Linear thermal expansion (10$^{-6}$ K$^{-1}$)")
plt.show()

{
    "alpha(300 K) in 1/K": float(alpha[i300, 0, 0]),
    "mean Grueneisen parameter at 300 K": float(
        np.trace(result.average_gruneisen[i300]) / 3
    ),
    "has imaginary modes": result.has_imaginary_modes,
}

With these settings we get $\alpha$(300 K) = 9.6e-6 K$^{-1}$ and a mean
Grüneisen parameter of 1.36. Experiment gives about 1.0e-5 K$^{-1}$ for MgO at
room temperature.

## Checking the fit

If the thermal expansion comes out as zero or very small, check the
third-order force constants first. All zeros means the LASSO fit dropped them.
The penalty chosen by cross validation should also lie inside the search
range, not on one of its bounds.

In [ ]:
import re
from pathlib import Path

import h5py

fit_dir = Path(doc.phonon_job_dir) / "one_shot"
log = (fit_dir / "pheasy_anharmonic_fit.log").read_text()
with h5py.File(fit_dir / "fc3.hdf5") as file:
    fc3 = file["fc3"][:]

{
    "LASSO penalty": re.findall(r"alpha_(?:min|max|opt):\s*\S+", log),
    "max |fc3| in eV/A^3": float(np.abs(fc3).max()),
}

## Known limitations

The frequencies and Grüneisen tensors are those of the relaxed structure. They
are not renormalized with temperature, so the result is least reliable at high
temperature.

A machine-learned potential gives no Born charges, so the non-analytical term
correction is not applied here. With VASP, the Born charges of the phonon run
are used.

If a frequency on the q-point mesh is below -0.1 THz, a warning is raised and
the thermal expansion of that fit is not computed.